In [1]:
from sqlalchemy import create_engine, text
import urllib.parse
import pandas as pd
import os, glob
import requests

In [2]:
params = urllib.parse.quote_plus(
    "DRIVER={ODBC Driver 18 for SQL Server};"
    "SERVER=localhost;"
    "DATABASE=fpl_fantasy;"
    "Trusted_Connection=yes;"
    "Encrypt=yes;"
    "TrustServerCertificate=yes;"
)

In [3]:
#connection to sql server + test to see if connection works
engine  = create_engine(f"mssql+pyodbc:///?odbc_connect={params}")

with engine.connect() as conn:
    result = conn.execute(text("SELECT @@VERSION;"))
    print(result.scalar())

C:\Users\JesseOnu\AppData\Local\Temp\ipykernel_32080\228086686.py:4: SAWarning: Unrecognized server version info '17.0.1000.7'.  Some SQL Server features may not function properly.
  with engine.connect() as conn:


Microsoft SQL Server 2025 (RTM) - 17.0.1000.7 (X64) 
	Oct 21 2025 12:05:57 
	Copyright (C) 2025 Microsoft Corporation
	Standard Developer Edition (64-bit) on Windows 10 Home 10.0 <X64> (Build 26200: )



In [4]:
def combine_upload_tosql(folderpath: str, tablename: str, engine):
    """
    Reads ALL CSV files in the folder and uploads them to SQL.
    Stacks multiple files into one table.
    """
    # Get all CSV files
    files = sorted(glob.glob(os.path.join(folderpath, "*.csv")))
    
    if not files:
        print(f"⚠️ No CSV files found in {folderpath}")
        return
    
    print(f"Found {len(files)} CSV files for table '{tablename}'")
    
    for i, path in enumerate(files):
        try:
            df = pd.read_csv(path, dtype=str, keep_default_na=False)
            
            # Replace on first file (i==0), append on all others
            mode = "replace" if i == 0 else "append"
            
            df.to_sql(name=tablename, 
                      con=engine, 
                      schema="dbo",
                      if_exists=mode,
                      index=False)
            
            print(f"   Uploaded: {os.path.basename(path)} ({len(df):,} rows)")
            
        except Exception as e:
            print(f"   ❌ Error with {os.path.basename(path)}: {e}")
    
    print(f"✅ Finished loading {tablename} from {len(files)} files")

In [ ]:
teampath = r"C:\Users\JesseOnu\fpl sql rework\teams"
teams = combine_upload_tosql(teampath, "stg_team", engine)
teams

Found 2 CSV files for table 'stg_team'


In [5]:
playerpath = r"C:\Users\JesseOnu\fpl sql rework\players"
players = combine_upload_tosql(playerpath, "stg_player", engine)
players

Found 2 CSV files for table 'stg_player'
   Uploaded: 2526playerid.csv (841 rows)
   Uploaded: 2627playerid.csv (623 rows)
✅ Finished loading stg_player from 2 files


In [5]:
scorepath = r"C:\Users\JesseOnu\fpl sql rework\playerscores"
playerscore = combine_upload_tosql(scorepath, "stg_player_scores", engine)
playerscore

Found 2 CSV files for table 'stg_player_scores'
   Uploaded: 2526player_scores.csv (11,492 rows)
   Uploaded: 2627player_scores.csv (622 rows)
✅ Finished loading stg_player_scores from 2 files


In [6]:
fixturepath = r"C:\Users\JesseOnu\fpl sql rework\fixtures"
fixture = combine_upload_tosql(fixturepath, "stg_fixtures", engine)
fixture

Found 2 CSV files for table 'stg_fixtures'
   Uploaded: 2526fixtures.csv (380 rows)
   Uploaded: 2627fixtures.csv (380 rows)
✅ Finished loading stg_fixtures from 2 files


In [7]:
fantasypath = r"C:\Users\JesseOnu\fpl sql rework\fantasygw"
fantasy = combine_upload_tosql(fantasypath, "stg_fantasy_gw", engine)
fantasy

Found 2 CSV files for table 'stg_fantasy_gw'
   Uploaded: 2526fantasygw.csv (38 rows)
   Uploaded: 2627fantasygw.csv (38 rows)
✅ Finished loading stg_fantasy_gw from 2 files


In [8]:
gwpath = r"C:\Users\JesseOnu\fpl sql rework\gws"
gw = combine_upload_tosql(gwpath, "stg_player_gameweek", engine)
gw

Found 40 CSV files for table 'stg_player_gameweek'
   Uploaded: 2526gw1.csv (690 rows)
   Uploaded: 2526gw10.csv (747 rows)
   Uploaded: 2526gw11.csv (752 rows)
   Uploaded: 2526gw12.csv (755 rows)
   Uploaded: 2526gw13.csv (755 rows)
   Uploaded: 2526gw14.csv (758 rows)
   Uploaded: 2526gw15.csv (759 rows)
   Uploaded: 2526gw16.csv (760 rows)
   Uploaded: 2526gw17.csv (770 rows)
   Uploaded: 2526gw18.csv (775 rows)
   Uploaded: 2526gw19.csv (780 rows)
   Uploaded: 2526gw2.csv (705 rows)
   Uploaded: 2526gw20.csv (790 rows)
   Uploaded: 2526gw21.csv (795 rows)
   Uploaded: 2526gw22.csv (799 rows)
   Uploaded: 2526gw23.csv (803 rows)
   Uploaded: 2526gw24.csv (811 rows)
   Uploaded: 2526gw25.csv (817 rows)
   Uploaded: 2526gw26.csv (896 rows)
   Uploaded: 2526gw27.csv (818 rows)
   Uploaded: 2526gw28.csv (819 rows)
   Uploaded: 2526gw29.csv (820 rows)
   Uploaded: 2526gw3.csv (712 rows)
   Uploaded: 2526gw30.csv (822 rows)
   Uploaded: 2526gw31.csv (825 rows)
   Uploaded: 2526gw32.csv (